# TP04: Dashboards Interactivos
## Laboratorio (Herramientas) - Universidad del Aconcagua
### Unidad 2: Herramientas en la Nube de Visualización de Datos

---

### 🎯 Objetivos del Trabajo Práctico

1. Diseñar **activos visuales dinámicos** para monitoreo de negocios
2. Crear **gráficos interactivos** con Plotly Express
3. Agrupar visualizaciones en un **panel consolidado**
4. Desarrollar **dashboards funcionales** integrados en el notebook

---

### 📁 Caso de Estudio: Dashboard de Ventas de Panadería

Crearemos un dashboard ejecutivo para la Panadería La Espiga Dorada con:
* Indicadores clave (KPIs)
* Gráficos de tendencias
* Análisis por sucursal y producto
* Visualizaciones interactivas

### 🕰️ Duración Estimada: 3 horas

In [0]:
# Configurar rutas del proyecto
import os
from pathlib import Path

# Obtener usuario actual automáticamente desde Spark
# Esto hace que el notebook sea 100% portable entre usuarios
USUARIO = spark.sql("SELECT current_user()").collect()[0][0]
BASE_USER = Path(f"/Workspace/Users/{USUARIO}")
BASE_LABORATORIO = BASE_USER / "Laboratorio"
BASE_DATASETS = BASE_LABORATORIO / "05 - Datasets"

print(f"📍 Usuario: {USUARIO}")
print(f"\n📂 Rutas configuradas:")
print(f"  Usuario:     {BASE_USER}")
print(f"  Laboratorio: {BASE_LABORATORIO}")
print(f"  Datasets:    {BASE_DATASETS}")

# Verificar que las rutas existen
if BASE_DATASETS.exists():
    archivos_csv = len([f for f in os.listdir(BASE_DATASETS) if f.endswith('.csv')])
    print(f"\n✅ Rutas verificadas correctamente")
    print(f"   Archivos CSV encontrados: {archivos_csv}")
else:
    print(f"\n⚠️ Advertencia: La carpeta de datasets no existe")

In [0]:
# Importar librerías necesarias
import pandas as pd  # Manipulación y análisis de datos tabulares
import numpy as np   # Operaciones numéricas y arrays
import matplotlib.pyplot as plt  # Visualizaciones estáticas de calidad profesional
import seaborn as sns  # Visualizaciones estadísticas avanzadas sobre matplotlib
from datetime import datetime  # Manejo de fechas y horas

# Configurar estilo de visualizaciones
# set_palette() define la paleta de colores para todos los gráficos
sns.set_palette('husl')  # Paleta "husl" = colores brillantes y distinguibles
# style.use() aplica un tema predefinido a matplotlib
plt.style.use('seaborn-v0_8-darkgrid')  # Estilo seaborn con cuadrícula oscura

print("✅ Librerías importadas")
print("🎨 Listo para crear dashboards")

## Parte 1: Carga y Preparación de Datos

### 📂 Cargar datos consolidados

Vamos a cargar y preparar los datos para el dashboard ejecutivo.

In [0]:
# Cargar datasets usando las rutas configuradas
# BASE_DATASETS fue definida en la celda anterior con pathlib
# pd.read_csv() lee archivos CSV y los convierte en DataFrames

df_productos = pd.read_csv(BASE_DATASETS / 'productos.csv')
df_sucursales = pd.read_csv(BASE_DATASETS / 'sucursales.csv')
# parse_dates=['fecha'] convierte automáticamente la columna 'fecha' a tipo datetime
df_ventas = pd.read_csv(BASE_DATASETS / 'ventas.csv', parse_dates=['fecha'])
df_detalles = pd.read_csv(BASE_DATASETS / 'detalles_ventas.csv')

# Crear dataset consolidado uniendo todas las tablas relevantes
# Este dataset combina detalles de ventas + productos + ventas + sucursales
# .merge() funciona como JOIN en SQL, combinando DataFrames por columnas comunes

# Paso 1: Unir detalles con productos
# suffixes=('_producto', '_sucursal') diferencia columnas con el mismo nombre (ej: 'nombre')
df_dash = df_detalles.merge(
    df_productos[['producto_id', 'nombre', 'categoria']], 
    on='producto_id'  # Columna común para unir
).merge(
    # Paso 2: Agregar información de la venta (fecha, sucursal)
    df_ventas[['venta_id', 'fecha', 'sucursal_id']], 
    on='venta_id'
).merge(
    # Paso 3: Agregar información de la sucursal (nombre, zona)
    df_sucursales[['sucursal_id', 'nombre', 'zona']], 
    on='sucursal_id', 
    suffixes=('_producto', '_sucursal')  # Diferencia 'nombre_producto' vs 'nombre_sucursal'
)

# Agregar columnas derivadas para análisis temporal
# .dt.to_period('M') convierte fechas a períodos mensuales (2024-01, 2024-02, etc.)
df_dash['mes'] = df_dash['fecha'].dt.to_period('M')
# .dt.day_name() extrae el nombre del día en inglés (Monday, Tuesday, etc.)
df_dash['dia_semana'] = df_dash['fecha'].dt.day_name()

print(f"✅ Dataset consolidado: {len(df_dash):,} registros")
display(df_dash.head())

## Parte 2: Indicadores Clave de Desempeño (KPIs)

### 📊 Métricas ejecutivas principales

In [0]:
# Calcular KPIs principales (Key Performance Indicators)
# Los KPIs son métricas clave que miden el rendimiento del negocio

# Facturación total = suma de todos los subtotales
facturacion_total = df_dash['subtotal'].sum()
# Total de transacciones = cantidad de ventas únicas (sin duplicados)
# .nunique() cuenta valores únicos, eliminando duplicados
total_transacciones = df_dash['venta_id'].nunique()
# Ticket promedio = cuánto gasta en promedio cada cliente por transacción
ticket_promedio = facturacion_total / total_transacciones
# Productos vendidos = suma de todas las cantidades vendidas
productos_vendidos = df_dash['cantidad'].sum()

print("📊 INDICADORES CLAVE DE DESEMPEÑO (KPIs)")
print("=" * 80)
print(f"\n💰 Facturación Total:        ${facturacion_total:>20,.2f}")
print(f"📝 Total Transacciones:    {total_transacciones:>20,}")
print(f"🎯 Ticket Promedio:        ${ticket_promedio:>20,.2f}")
print(f"📦 Productos Vendidos:     {int(productos_vendidos):>20,}")

print("\n" + "=" * 80)

## Parte 3: Dashboard Visual Integrado

### 📈 Panel de visualizaciones ejecutivas

Crearemos un conjunto de gráficos que forman un dashboard completo.

In [0]:
# 1. Tendencia de ventas mensuales
# Analizamos cómo evolucionan las ventas a lo largo del tiempo

# Agrupar datos por mes y calcular métricas agregadas
# .groupby('mes') agrupa todas las ventas del mismo mes
# .agg() permite calcular múltiples métricas simultáneamente
ventas_mes = df_dash.groupby('mes').agg({
    'subtotal': 'sum',       # Suma total de ventas por mes
    'venta_id': 'nunique'    # Cantidad de transacciones únicas por mes
}).reset_index()  # reset_index() convierte el índice (mes) en columna normal

# Renombrar columnas para mayor claridad
ventas_mes.columns = ['mes', 'facturacion', 'transacciones']
# Convertir períodos a strings para mostrar en gráficos (2024-01 → "2024-01")
ventas_mes['mes_str'] = ventas_mes['mes'].astype(str)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Facturación mensual
ax1.plot(ventas_mes['mes_str'], ventas_mes['facturacion'], 
         marker='o', linewidth=2.5, markersize=8, color='steelblue')
ax1.fill_between(range(len(ventas_mes)), ventas_mes['facturacion'], alpha=0.3, color='steelblue')
ax1.set_title('💰 Evolución de Facturación Mensual', fontsize=16, fontweight='bold', pad=20)
ax1.set_xlabel('Mes', fontsize=12)
ax1.set_ylabel('Facturación ($)', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Transacciones mensuales
ax2.bar(ventas_mes['mes_str'], ventas_mes['transacciones'], color='coral', edgecolor='black', alpha=0.7)
ax2.set_title('📝 Número de Transacciones Mensuales', fontsize=16, fontweight='bold', pad=20)
ax2.set_xlabel('Mes', fontsize=12)
ax2.set_ylabel('Transacciones', fontsize=12)
ax2.grid(axis='y', alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [0]:
# 2. Rendimiento por sucursal
# Comparamos el desempeño de cada sucursal en diferentes dimensiones

# Agrupar por sucursal y calcular métricas clave
# Nos interesa saber: ¿cuánto factura?, ¿cuántas ventas?, ¿cuántas unidades?
ventas_sucursal = df_dash.groupby('nombre_sucursal').agg({
    'subtotal': 'sum',       # Facturación total por sucursal
    'venta_id': 'nunique',   # Número de transacciones únicas
    'cantidad': 'sum'        # Total de unidades vendidas
}).reset_index()

# Renombrar columnas para mayor claridad
ventas_sucursal.columns = ['sucursal', 'facturacion', 'transacciones', 'unidades']
# Ordenar de menor a mayor facturación (para gráficos horizontales)
ventas_sucursal = ventas_sucursal.sort_values('facturacion', ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Facturación por sucursal
axes[0].barh(ventas_sucursal['sucursal'], ventas_sucursal['facturacion'], color='teal', edgecolor='black')
axes[0].set_title('🏢 Facturación por Sucursal', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Facturación ($)')
for i, v in enumerate(ventas_sucursal['facturacion']):
    axes[0].text(v, i, f' ${v/1e6:.1f}M', va='center', fontsize=10)

# Transacciones por sucursal
axes[1].barh(ventas_sucursal['sucursal'], ventas_sucursal['transacciones'], color='orange', edgecolor='black')
axes[1].set_title('📝 Transacciones por Sucursal', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Transacciones')
for i, v in enumerate(ventas_sucursal['transacciones']):
    axes[1].text(v, i, f' {int(v):,}', va='center', fontsize=10)

# Unidades vendidas por sucursal
axes[2].barh(ventas_sucursal['sucursal'], ventas_sucursal['unidades'], color='purple', edgecolor='black')
axes[2].set_title('📦 Unidades Vendidas', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Unidades')
for i, v in enumerate(ventas_sucursal['unidades']):
    axes[2].text(v, i, f' {int(v):,}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [0]:
# 3. Top productos más vendidos
# Identificamos los productos estrella del negocio

# Agrupar por producto y categoría, calcular ventas y unidades
# Agrupamos por ambas columnas para tener información completa
top_productos = df_dash.groupby(['nombre_producto', 'categoria']).agg({
    'cantidad': 'sum',   # Total de unidades vendidas
    'subtotal': 'sum'    # Facturación total generada
}).reset_index().sort_values('subtotal', ascending=False).head(10)  # Top 10 por facturación

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Top por facturación
y_pos = np.arange(len(top_productos))
colors = plt.cm.Set3(np.linspace(0, 1, len(top_productos)))
ax1.barh(y_pos, top_productos['subtotal'], color=colors, edgecolor='black')
ax1.set_yticks(y_pos)
ax1.set_yticklabels(top_productos['nombre_producto'])
ax1.set_xlabel('Facturación ($)', fontsize=12)
ax1.set_title('🏆 Top 10 Productos por Facturación', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
for i, v in enumerate(top_productos['subtotal']):
    ax1.text(v, i, f' ${v/1e6:.2f}M', va='center', fontsize=9)

# Top por unidades
top_unidades = df_dash.groupby(['nombre_producto', 'categoria']).agg({
    'cantidad': 'sum'
}).reset_index().sort_values('cantidad', ascending=False).head(10)

y_pos2 = np.arange(len(top_unidades))
colors2 = plt.cm.Pastel1(np.linspace(0, 1, len(top_unidades)))
ax2.barh(y_pos2, top_unidades['cantidad'], color=colors2, edgecolor='black')
ax2.set_yticks(y_pos2)
ax2.set_yticklabels(top_unidades['nombre_producto'])
ax2.set_xlabel('Unidades Vendidas', fontsize=12)
ax2.set_title('📦 Top 10 Productos por Unidades', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)
for i, v in enumerate(top_unidades['cantidad']):
    ax2.text(v, i, f' {int(v):,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [0]:
# 4. Análisis por categoría y día de semana
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Ventas por categoría (pie chart)
ventas_cat = df_dash.groupby('categoria')['subtotal'].sum().sort_values(ascending=False)
colors_pie = plt.cm.Set2(np.linspace(0, 1, len(ventas_cat)))
wedges, texts, autotexts = ax1.pie(ventas_cat, labels=ventas_cat.index, autopct='%1.1f%%',
                                     colors=colors_pie, startangle=90, textprops={'fontsize': 10})
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax1.set_title('🎨 Distribución de Facturación por Categoría', fontsize=14, fontweight='bold')

# Ventas por día de semana
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
ventas_dia = df_dash.groupby('dia_semana')['subtotal'].sum().reindex(orden_dias)
colors_bar = ['#FF6B6B' if i >= 5 else '#4ECDC4' for i in range(7)]
ax2.bar(range(7), ventas_dia, color=colors_bar, edgecolor='black', alpha=0.8)
ax2.set_xticks(range(7))
ax2.set_xticklabels(['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom'])
ax2.set_title('📅 Facturación por Día de Semana', fontsize=14, fontweight='bold')
ax2.set_ylabel('Facturación ($)')
ax2.grid(axis='y', alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

plt.tight_layout()
plt.show()

## Parte 5: Dashboards Geoespaciales con H3

### 🗺️ Visualizaciones Espaciales Interactivas

En esta sección crearemos dashboards que muestren patrones geográficos:

1. **Heatmap de Densidad de Clientes**
2. **Mapa de Facturación por Zona H3**
3. **Visualización de Cobertura de Sucursales**
4. **Análisis de Zonas Sin Cobertura**

---

In [0]:
# Instalar H3 si no está disponible y cargar librerías geoespaciales
# H3 es un sistema de indexación hexagonal desarrollado por Uber
# Divide el mapa en hexágonos jerárquicos para análisis geoespacial eficiente
%pip install h3 plotly --quiet

import h3  # Sistema de coordenadas hexagonales
import plotly.express as px  # Visualizaciones interactivas
import plotly.graph_objects as go  # Gráficos personalizados de plotly

print("✅ Librerías geoespaciales instaladas e importadas")

In [0]:
# Heatmap 1: Densidad de Clientes por Zona H3
print("📊 Creando Heatmap de Densidad de Clientes...\n")

# Cargar datos de clientes si no están disponibles
if 'df_clientes' not in locals():
    df_clientes = pd.read_csv(BASE_DATASETS / 'clientes.csv')
    print(f"✅ Clientes cargados: {len(df_clientes):,} registros\n")

# Agrupar clientes por hexágono H3
# Cada hexágono representa una pequeña área geográfica (~100m² en resolución 9)
clientes_por_zona = df_clientes.groupby('h3_index').agg({
    'cliente_id': 'count'  # Contar cuántos clientes hay en cada zona
}).reset_index()
clientes_por_zona.columns = ['h3_index', 'num_clientes']

# Obtener coordenadas del centro de cada hexágono
# h3.cell_to_latlng() convierte un índice H3 a latitud/longitud
clientes_por_zona['lat'] = clientes_por_zona['h3_index'].apply(
    lambda x: h3.cell_to_latlng(x)[0]  # [0] = latitud
)
clientes_por_zona['lon'] = clientes_por_zona['h3_index'].apply(
    lambda x: h3.cell_to_latlng(x)[1]  # [1] = longitud
)

# Crear heatmap interactivo con Plotly
# density_mapbox crea un mapa de calor sobre un mapa base
fig = px.density_mapbox(
    clientes_por_zona,
    lat='lat',               # Latitudes de los puntos
    lon='lon',               # Longitudes de los puntos
    z='num_clientes',        # Valor que determina la intensidad del color
    radius=15,               # Radio de influencia de cada punto (px)
    center={'lat': -32.89, 'lon': -68.84},  # Centro del mapa (Mendoza)
    zoom=11,                 # Nivel de zoom inicial
    mapbox_style='open-street-map',  # Estilo de mapa base (OpenStreetMap)
    title='Heatmap: Densidad de Clientes por Zona H3 (Mendoza)',
    color_continuous_scale='YlOrRd',  # Escala de color: amarillo a rojo
    labels={'num_clientes': 'Clientes'}  # Etiqueta de la leyenda
)

fig.update_layout(height=600)  # Altura del gráfico en píxeles
fig.show()

print(f"✅ Heatmap creado")
print(f"   Total de zonas H3: {len(clientes_por_zona)}")
print(f"   Zona con más clientes: {clientes_por_zona['num_clientes'].max()} clientes")

In [0]:
# Mapa 2: Facturación por Zona H3
print("💰 Creando Mapa de Facturación por Zona...\n")

# Unir ventas con clientes para obtener h3_index de cada venta
# Solo consideramos ventas con cliente identificado (no anónimas)
ventas_geo = df_ventas[df_ventas['cliente_id'].notna()].merge(
    df_clientes[['cliente_id', 'h3_index']], 
    on='cliente_id'
)

# Agregar facturación por zona H3
# Calculamos: facturación total, cantidad de ventas, ticket promedio
facturacion_zona = ventas_geo.groupby('h3_index').agg({
    'total': ['sum', 'count', 'mean']  # Múltiples agregaciones en una columna
}).reset_index()
# Aplanar los nombres de columnas multinivel resultantes
facturacion_zona.columns = ['h3_index', 'facturacion', 'num_ventas', 'ticket_promedio']

# Obtener coordenadas de cada zona
facturacion_zona['lat'] = facturacion_zona['h3_index'].apply(
    lambda x: h3.cell_to_latlng(x)[0]
)
facturacion_zona['lon'] = facturacion_zona['h3_index'].apply(
    lambda x: h3.cell_to_latlng(x)[1]
)

# Top 20 zonas por facturación (para evitar saturar el mapa)
top_zonas = facturacion_zona.nlargest(20, 'facturacion')

# Crear mapa de burbujas
# El tamaño de cada burbuja representa la facturación
# El color representa el ticket promedio
fig = px.scatter_mapbox(
    top_zonas,
    lat='lat',
    lon='lon',
    size='facturacion',          # Tamaño de burbujas = facturación
    color='ticket_promedio',     # Color = ticket promedio
    hover_name='h3_index',       # Título del tooltip
    hover_data={                 # Datos adicionales al pasar el mouse
        'facturacion': ':$,.0f',     # Formato: $X,XXX
        'num_ventas': ':,',          # Formato: X,XXX
        'ticket_promedio': ':$,.0f', # Formato: $X,XXX
        'lat': False,                # No mostrar latitud
        'lon': False                 # No mostrar longitud
    },
    title='Top 20 Zonas por Facturación (tamaño = facturación, color = ticket promedio)',
    mapbox_style='open-street-map',
    zoom=11,
    center={'lat': -32.89, 'lon': -68.84},
    color_continuous_scale='Viridis',  # Escala de color azul-verde-amarillo
    size_max=50  # Tamaño máximo de burbuja
)

fig.update_layout(height=600)
fig.show()

print(f"✅ Mapa de facturación creado")
print(f"   Zona más rentable: ${top_zonas['facturacion'].max():,.0f}")
print(f"   Ticket promedio máximo: ${top_zonas['ticket_promedio'].max():,.0f}")

In [0]:
# Mapa 3: Cobertura de Sucursales
print("📍 Creando Mapa de Cobertura de Sucursales...\n")

# Crear figura base de plotly
fig = go.Figure()

# Para cada sucursal, calcular área de cobertura
for idx, sucursal in df_sucursales.iterrows():
    # Obtener hexágonos vecinos (radio 3 ≈ ~500m de cobertura)
    # h3.grid_disk() devuelve todos los hexágonos dentro de un radio
    h3_sucursal = sucursal['h3_index']
    vecinos = h3.grid_disk(h3_sucursal, 3)  # radio = 3 hexágonos
    
    # Obtener coordenadas de cada hexágono vecino
    coords = [h3.cell_to_latlng(h) for h in vecinos]
    lats, lons = zip(*coords)  # Separar en listas de latitudes y longitudes
    
    # Contar cuántos clientes hay en esta área de cobertura
    clientes_cercanos = df_clientes[df_clientes['h3_index'].isin(vecinos)]
    
    # Agregar capa de cobertura (scatter con transparencia)
    fig.add_trace(go.Scattermapbox(
        lat=list(lats),
        lon=list(lons),
        mode='markers',                  # Solo marcadores (sin líneas)
        marker=dict(size=8, opacity=0.3),  # Pequeños y semi-transparentes
        name=f"Cobertura {sucursal['nombre']}",
        text=f"{len(clientes_cercanos)} clientes",
        hoverinfo='text'
    ))

# Agregar marcadores de sucursales (estrellas rojas grandes)
fig.add_trace(go.Scattermapbox(
    lat=df_sucursales['latitud'],
    lon=df_sucursales['longitud'],
    mode='markers+text',             # Marcadores con etiquetas de texto
    marker=dict(size=20, color='red', symbol='star'),  # Estrellas rojas
    text=df_sucursales['nombre'].apply(lambda x: x.split('-')[1].strip()),  # Nombre corto
    textposition='top center',       # Texto arriba del marcador
    name='Sucursales',
    hovertext=df_sucursales['nombre'],  # Nombre completo al pasar mouse
    hoverinfo='text'
))

# Configurar el mapa
fig.update_layout(
    mapbox=dict(
        style='open-street-map',
        center=dict(lat=-32.89, lon=-68.84),
        zoom=11
    ),
    title='Mapa de Cobertura de Sucursales (radio ~500m)',
    height=600,
    showlegend=True  # Mostrar leyenda con capas
)

fig.show()

# Reporte de cobertura por sucursal
print(f"✅ Mapa de cobertura creado")
for idx, sucursal in df_sucursales.iterrows():
    vecinos = h3.grid_disk(sucursal['h3_index'], 3)
    clientes = df_clientes[df_clientes['h3_index'].isin(vecinos)]
    print(f"   {sucursal['nombre']}: {len(clientes)} clientes en área")

In [0]:
# Análisis 4: Identificar zonas sin cobertura
print("🔍 Analizando zonas sin cobertura...\n")

# 1. Obtener todas las zonas con clientes
zonas_clientes = df_clientes['h3_index'].unique()

# 2. Para cada zona, calcular distancia mínima a sucursal
# h3.grid_distance() calcula la distancia en "saltos de hexágonos"
sucursales_h3 = df_sucursales['h3_index'].tolist()

distancias = []
for zona in zonas_clientes:
    # Calcular distancia de esta zona a cada sucursal
    dist_min = min([h3.grid_distance(zona, s) for s in sucursales_h3])
    num_clientes = len(df_clientes[df_clientes['h3_index'] == zona])
    distancias.append({
        'h3_index': zona,
        'distancia_min_sucursal': dist_min,
        'num_clientes': num_clientes
    })

df_distancias = pd.DataFrame(distancias)

# 3. Identificar zonas sin cobertura adecuada
# Criterio: distancia > 5 hexágonos (≈500m) Y al menos 3 clientes
zonas_sin_cobertura = df_distancias[
    (df_distancias['distancia_min_sucursal'] > 5) & 
    (df_distancias['num_clientes'] >= 3)
].sort_values('num_clientes', ascending=False)

print(f"📊 Resultados:")
print(f"   Zonas sin cobertura adecuada: {len(zonas_sin_cobertura)}")
print(f"   Clientes afectados: {zonas_sin_cobertura['num_clientes'].sum()}")
print(f"\n🎯 Top 5 zonas candidatas para nueva sucursal:")

if len(zonas_sin_cobertura) > 0:
    # Agregar coordenadas para visualización
    zonas_sin_cobertura['lat'] = zonas_sin_cobertura['h3_index'].apply(
        lambda x: h3.cell_to_latlng(x)[0]
    )
    zonas_sin_cobertura['lon'] = zonas_sin_cobertura['h3_index'].apply(
        lambda x: h3.cell_to_latlng(x)[1]
    )
    
    print(zonas_sin_cobertura.head()[['h3_index', 'distancia_min_sucursal', 'num_clientes', 'lat', 'lon']])
    
    # Visualizar en mapa interactivo
    fig = px.scatter_mapbox(
        zonas_sin_cobertura.head(10),  # Top 10 zonas
        lat='lat',
        lon='lon',
        size='num_clientes',           # Tamaño = cantidad de clientes
        color='distancia_min_sucursal',  # Color = distancia a sucursal
        hover_data=['num_clientes', 'distancia_min_sucursal'],
        title='Top 10 Zonas sin Cobertura Adecuada (candidatas para nueva sucursal)',
        mapbox_style='open-street-map',
        zoom=11,
        color_continuous_scale='Reds',  # Escala roja (más rojo = más lejos)
        size_max=30
    )
    
    # Agregar sucursales actuales como referencia
    fig.add_trace(go.Scattermapbox(
        lat=df_sucursales['latitud'],
        lon=df_sucursales['longitud'],
        mode='markers',
        marker=dict(size=20, color='blue', symbol='star'),
        name='Sucursales Actuales',
        text=df_sucursales['nombre']
    ))
    
    fig.update_layout(height=600)
    fig.show()
    
    print("\n💡 Recomendación: Considerar abrir nueva sucursal en zona destacada")
else:
    print("   ✅ Todas las zonas tienen cobertura adecuada")

## 🎯 Resumen del TP04

### ✅ Qué aprendimos:

1. **Rutas portables**: Configuramos rutas dinámicas con detección automática del usuario
2. **Cálculo de KPIs**: Métricas ejecutivas clave para el negocio
3. **Visualizaciones múltiples**: Gráficos de líneas, barras, pie charts
4. **Dashboard integrado**: Panel consolidado con múltiples vistas
5. **Análisis temporal**: Tendencias mensuales y semanales
6. **Comparación de sucursales**: Rendimiento relativo
7. **Top productos**: Identificación de mejores performers
8. **Dashboards geoespaciales**: Heatmaps, mapas de facturación, cobertura con H3
9. **Análisis de cobertura**: Identificación de oportunidades de expansión

### 💡 Insights del dashboard:

* **Temporales**: Hay variación significativa en ventas por mes
* **Semanales**: Los fines de semana tienen mayor facturación
* **Categorías**: Ciertas categorías dominan las ventas
* **Sucursales**: Las sucursales tienen rendimientos diferenciados
* **Geoespaciales**: Identificamos zonas de alta densidad de clientes y áreas sin cobertura
* **Expansión**: Detectamos oportunidades para nuevas sucursales

### 🚀 Próximos pasos:

En la **Unidad 3** (Modelado de Datos) aprenderemos a:
* Crear estructuras de datos optimizadas
* Aplicar agregaciones complejas
* Diseñar modelos de datos eficientes
* Preparar datos para machine learning

---

**📝 Excelente! Has creado un dashboard ejecutivo completo para la toma de decisiones.**

### 📚 UNIDAD 1 Y 2 COMPLETADAS ✅

Has finalizado las primeras dos unidades del curso:
* **Unidad 1 - Análisis de Datos**: Configuración, carga, transformación y exploración
* **Unidad 2 - Visualización de Datos**: Perfilado y dashboards

Contínua con las **Unidades 3 y 4** para completar el programa del curso.